In [ ]:
import pandas as pd

df = pd.read_csv('chocolate_sales_2025_dataset.csv')
df.head()

,Sale_ID,Date,Brand,Product_Type,Country,Sales_Channel,Payment_Method,Price_USD,Units_Sold,Revenue_USD
0,1,2025-11-24,Cadbury,Milk Chocolate,France,Supermarket,Digital Wallet,5.00,194,970.00
1,2,2025-02-22,Lindt,Chocolate Bar,India,Online,Cash,17.73,144,2553.12
2,3,2025-02-17,Toblerone,Dark Chocolate,Australia,Supermarket,Digital Wallet,7.42,134,994.28
3,4,2025-11-29,Ferrero,Truffles,Italy,Convenience Store,Cash,18.28,112,2047.36
4,5,2025-03-23,Cadbury,Milk Chocolate,France,Convenience Store,Cash,18.21,92,1675.32


In [ ]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sale_ID         500 non-null    int64  
 1   Date            500 non-null    object 
 2   Brand           500 non-null    object 
 3   Product_Type    500 non-null    object 
 4   Country         500 non-null    object 
 5   Sales_Channel   500 non-null    object 
 6   Payment_Method  500 non-null    object 
 7   Price_USD       500 non-null    float64
 8   Units_Sold      500 non-null    int64  
 9   Revenue_USD     500 non-null    float64
dtypes: float64(2), int64(2), object(6)
memory usage: 39.2+ KB


,Sale_ID,Price_USD,Units_Sold,Revenue_USD
count,500.000000,500.000000,500.000000,500.000000
mean,250.500000,13.779860,104.938000,1433.391140
std,144.481833,6.484013,56.263998,1065.679386
min,1.000000,2.520000,5.000000,20.480000
25%,125.750000,8.592500,56.750000,562.545000
50%,250.500000,13.480000,108.500000,1197.225000
75%,375.250000,19.445000,150.250000,2072.812500
max,500.000000,25.000000,200.000000,4809.260000


In [ ]:
print(df.columns)

Index(['Sale_ID', 'Date', 'Brand', 'Product_Type', 'Country', 'Sales_Channel',
       'Payment_Method', 'Price_USD', 'Units_Sold', 'Revenue_USD'],
      dtype='object')


In [ ]:
# ETL PROCESS - FINAL FIXED

import pandas as pd

# 1. EXTRACT
print("=== EXTRACT DATA ===")

df = pd.read_csv('chocolate_sales_2025_dataset.csv')
print(df.head())


# 2. TRANSFORM
print("\n=== TRANSFORM DATA ===")

# Bersihin nama kolom
df.columns = df.columns.str.strip()

# Convert Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Generate kolom waktu (FIX ERROR UTAMA)
df['Day'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Hapus data yang tanggalnya gagal convert
df = df.dropna(subset=['Date'])

# Hapus duplikat
df = df.drop_duplicates()

print("Kolom setelah transform:")
print(df.columns)


# 3. LOAD (DATA WAREHOUSE)
print("\n=== LOAD DATA ===")

#  DIMENSION

# Dim Product
dim_product = df[['Product_Type', 'Brand']].drop_duplicates().reset_index(drop=True)
dim_product['product_id'] = dim_product.index + 1

# Dim Region
dim_region = df[['Country']].drop_duplicates().reset_index(drop=True)
dim_region['region_id'] = dim_region.index + 1

# Dim Date
dim_date = df[['Date', 'Day', 'Month', 'Year']].drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = dim_date.index + 1

# Dim Sales Channel
dim_channel = df[['Sales_Channel']].drop_duplicates().reset_index(drop=True)
dim_channel['channel_id'] = dim_channel.index + 1

# Dim Payment
dim_payment = df[['Payment_Method']].drop_duplicates().reset_index(drop=True)
dim_payment['payment_id'] = dim_payment.index + 1


#  FACT TABLE
fact_sales = df[['Date', 'Product_Type', 'Brand', 'Country',
                 'Sales_Channel', 'Payment_Method',
                 'Revenue_USD', 'Units_Sold']]

# Join ke semua dimension
fact_sales = fact_sales.merge(dim_product, on=['Product_Type', 'Brand'], how='left')
fact_sales = fact_sales.merge(dim_region, on='Country', how='left')
fact_sales = fact_sales.merge(dim_date, on='Date', how='left')
fact_sales = fact_sales.merge(dim_channel, on='Sales_Channel', how='left')
fact_sales = fact_sales.merge(dim_payment, on='Payment_Method', how='left')

# Final fact table
fact_sales = fact_sales[[
    'date_id', 'product_id', 'region_id',
    'channel_id', 'payment_id',
    'Revenue_USD', 'Units_Sold'
]]


# 4. SAVE
print("\n=== SAVE DATA ===")

fact_sales.to_csv('fact_sales.csv', index=False)
dim_product.to_csv('dim_product.csv', index=False)
dim_region.to_csv('dim_region.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
dim_channel.to_csv('dim_channel.csv', index=False)
dim_payment.to_csv('dim_payment.csv', index=False)

print("Semua tabel berhasil disimpan!")


# 5. VALIDASI
print("\n=== VALIDASI ===")

print("\nFact Table:")
print(fact_sales.head())

print("\nDim Product:")
print(dim_product.head())

print("\nDim Region:")
print(dim_region.head())

print("\nDim Date:")
print(dim_date.head())

print("\n=== ETL SELESAI ===")

=== EXTRACT DATA ===
   Sale_ID        Date      Brand    Product_Type    Country  \
0        1  2025-11-24    Cadbury  Milk Chocolate     France   
1        2  2025-02-22      Lindt   Chocolate Bar      India   
2        3  2025-02-17  Toblerone  Dark Chocolate  Australia   
3        4  2025-11-29    Ferrero        Truffles      Italy   
4        5  2025-03-23    Cadbury  Milk Chocolate     France   

       Sales_Channel  Payment_Method  Price_USD  Units_Sold  Revenue_USD  
0        Supermarket  Digital Wallet       5.00         194       970.00  
1             Online            Cash      17.73         144      2553.12  
2        Supermarket  Digital Wallet       7.42         134       994.28  
3  Convenience Store            Cash      18.28         112      2047.36  
4  Convenience Store            Cash      18.21          92      1675.32  

=== TRANSFORM DATA ===
Kolom setelah transform:
Index(['Sale_ID', 'Date', 'Brand', 'Product_Type', 'Country', 'Sales_Channel',
       'Payment_

In [ ]:
print("📊 Key Insights\n")
print("Berikut insight utama yang berhasil ditemukan dari analisis data:\n")

# --- 1. Produk Terlaris ---
print("🥇 Produk Terlaris (Berdasarkan Revenue)")
produk = df.groupby('Product_Type')['Revenue_USD'].sum().sort_values(ascending=False).reset_index()
produk.index = produk.index + 1
produk.columns = ['Produk', 'Total Revenue']
# Formatting mata uang
produk['Total Revenue'] = produk['Total Revenue'].apply(lambda x: f"${x:,.2f}")
print(produk.head(5).to_markdown())

# --- 2. Pasar Terkuat per Negara ---
print("---")
print("🌍 Pasar Terkuat per Negara")
negara = df.groupby('Country')['Revenue_USD'].sum().sort_values(ascending=False).reset_index()
negara.index = negara.index + 1
negara.columns = ['Negara', 'Total Revenue']
negara['Total Revenue'] = negara['Total Revenue'].apply(lambda x: f"${x:,.2f}")
print(negara.head(5).to_markdown())

# --- 3. Saluran Penjualan ---
print("---")
print("🛒 Saluran Penjualan Paling Efektif")
saluran = df.groupby('Sales_Channel')['Revenue_USD'].sum().sort_values(ascending=False).reset_index()
total_rev = df['Revenue_USD'].sum()
saluran['Persentase'] = (saluran['Revenue_USD'] / total_rev * 100).map("{:.1f}%".format)
saluran['Revenue_USD'] = saluran['Revenue_USD'].apply(lambda x: f"${x:,.2f}")
saluran.columns = ['Saluran', 'Total Revenue', 'Persentase']
print(saluran.to_markdown(index=False))

# --- 4. Metode Pembayaran ---
print("---")
print("💳 Metode Pembayaran Favorit")
payment = df.groupby('Payment_Method')['Revenue_USD'].sum().sort_values(ascending=False).reset_index()
payment['Dominasi'] = (payment['Revenue_USD'] / total_rev * 100).map("{:.1f}%".format)
payment['Revenue_USD'] = payment['Revenue_USD'].apply(lambda x: f"${x:,.2f}")
payment.columns = ['Metode', 'Total Revenue', 'Dominasi']
print(payment.to_markdown(index=False))

# --- 5. Brand Terkuat ---
print("---")
print("🏆 Brand Terkuat")
brand = df.groupby('Brand')['Revenue_USD'].sum().sort_values(ascending=False).reset_index()
brand.index = brand.index + 1
brand.columns = ['Brand', 'Total Revenue']
brand['Total Revenue'] = brand['Total Revenue'].apply(lambda x: f"${x:,.2f}")
print(brand.head(5).to_markdown())

# --- 6. Ringkasan Keseluruhan ---
print("---")
print("📈 Ringkasan Keseluruhan\n")
summary = {
    "Total Revenue": f"${df['Revenue_USD'].sum():,.2f}",
    "Total Units Sold": f"{df['Units_Sold'].sum():,} unit",
    "Periode Data": f"{df['Date'].min().strftime('%b %Y')} — {df['Date'].max().strftime('%b %Y')}",
    "Cakupan Negara": f"{df['Country'].nunique()} negara",
    "Jumlah Brand": f"{df['Brand'].nunique()} brand",
    "Jenis Produk": f"{df['Product_Type'].nunique()} kategori"
}

for key, value in summary.items():
    print(f"{key.ljust(20)} :  {value}")

📊 Key Insights

Berikut insight utama yang berhasil ditemukan dari analisis data:

🥇 Produk Terlaris (Berdasarkan Revenue)
|    | Produk         | Total Revenue   |
|---:|:---------------|:----------------|
|  1 | Chocolate Box  | $151,378.86     |
|  2 | Milk Chocolate | $119,985.58     |
|  3 | Truffles       | $116,117.81     |
|  4 | Chocolate Bar  | $114,664.97     |
|  5 | Dark Chocolate | $108,655.80     |
---
🌍 Pasar Terkuat per Negara
|    | Negara   | Total Revenue   |
|---:|:---------|:----------------|
|  1 | France   | $89,956.88      |
|  2 | Canada   | $80,937.24      |
|  3 | India    | $77,033.35      |
|  4 | UK       | $76,796.23      |
|  5 | Germany  | $74,267.03      |
---
🛒 Saluran Penjualan Paling Efektif
| Saluran           | Total Revenue   | Persentase   |
|:------------------|:----------------|:-------------|
| Convenience Store | $200,001.57     | 27.9%        |
| Specialty Store   | $180,633.82     | 25.2%        |
| Supermarket       | $179,683.38     | 2